# How to Save and Export Cartogram Results

Once you have a cartogram you're happy with, you'll want to either save it for later or export it to a GIS format. This guide shows both options:

1. **Save and reload** — preserves the full `Cartogram` object (errors, history, options)
2. **Export to GeoDataFrame** — converts morphed geometries to a table you can plot or save to file

In [ ]:
import carto_flow.data as examples
import carto_flow.flow_cartogram as flow

In [ ]:
gdf = examples.load_us_census(population=True)
cartogram = flow.morph_gdf(gdf, "Population")

## Option 1: Save and reload the Cartogram object

`cartogram.save()` writes the result to a JSON file that can be reloaded with `Cartogram.load()`. This preserves everything: morphed geometries, error history, convergence data, and the options used.

In [ ]:
cartogram.save("my_cartogram.json")

# Later, in another session:
loaded = flow.Cartogram.load("my_cartogram.json")
print("Status after reload:", loaded.status)
print("Mean error:", loaded.get_errors().mean_error_pct, "%")

## Option 2: Export to GeoDataFrame

`to_geodataframe()` returns the morphed geometries as a GeoDataFrame with the original columns plus error and density columns. From there, you can write to any format GeoPandas supports.

In [ ]:
result_gdf = cartogram.to_geodataframe()
print(result_gdf.columns.tolist())
result_gdf.head(3)

In [ ]:
# Write to GeoPackage for use in QGIS, ArcGIS, etc.
result_gdf.to_file("cartogram.gpkg", driver="GPKG")

# Or to GeoJSON for web maps
result_gdf.to_file("cartogram.geojson", driver="GeoJSON")

## Quick check: plot the reloaded result

Verify the save/load round-trip by plotting the reloaded cartogram.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cartogram.plot("Population (Millions)", ax=axes[0], cmap="RdYlGn_r", legend=True, vmin=0, vmax=40)
axes[0].set(title="Original")
axes[0].axis("off")

loaded.to_geodataframe().plot("Population (Millions)", ax=axes[1], cmap="RdYlGn_r", legend=True, vmin=0, vmax=40)
axes[1].set(title="Cartogram (reloaded)")
axes[1].axis("off")

plt.tight_layout();